# 11 — best_model (feature-union v1): a single model on the union of the best M1-M5 features

**Not the vote ensemble** (which combines only the *scores*). best_model's edge = learning **cross-angle interactions** at the feature level (which a score vote cannot). Pool = **gate-passed + cross-model de-duplicated** features (Spearman>0.90) from M1-M5, angle-prefixed.

**Rigor (generous compute):** **shadow** features (leak detector), **stability selection 300x**, k selection by **per-fold permutation importance** (captures interactions, not just |d|), grid **under a gap cap**, **repeated nested CV** (unbiased AP + variance), **leave-one-dataset-out** (external validity), **SHAP** (which angles weigh) + confounders.

**Judge: OOF folds 1-8; fold10 NEVER touched until the final showdown, STRICT PRE-REGISTERED rule** (best_model wins only if it beats M4-alone AND the vote-ensemble by more than half the CI width; otherwise parsimony).

> DATA: reads the gate-passed columns of the `m1/m2/m3/m4/m5` feature CSVs (large files, heavy I/O once -> cached `bestmodel_pool.csv`). All blocks are guarded/resumable.

### Block bm.0: Setup + per-model gate-passed lists + rigor config

In [ ]:
# best_model — single XGBoost on the UNION of gate-passed features from M1..M5 (cross-model deduped).
# NOT the score-ensemble (that is the final ensemble step). Its edge = cross-angle FEATURE INTERACTIONS a score-vote can't see.
# Rigor (compute is available): shadow/null features (leak detector), stability selection 300x, per-fold
# permutation-importance k-selection (catches interaction features, not just univariate |d|), gap-capped grid,
# REPEATED nested-CV (unbiased AP + CI), leave-one-dataset-out external validity, SHAP + confounds.
# Judge: OOF folds 1-8; fold10 UNTOUCHED, pre-registered STRICT rule (beat M4-alone AND vote-ensemble by > half-CI).
import os, sys, json, warnings, time
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from tqdm import tqdm
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
warnings.filterwarnings('ignore'); EPS=1e-9
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); MODELS=os.path.join(ROOT,'models','best_model')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics'); SRC=os.path.join(ROOT,'src')
for d in (MODELS,FIG,METRICS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC); from evaluation import evaluate_standard
# ---- config ----
SHADOW_N=50           # random null features injected -> if selected, the selection is leaking
STAB_B=300; STAB_THR=0.50
GAP_CAP=0.30; TIE_EPS=0.01
NESTED_REPEATS=5      # repeated nested-CV for an unbiased AP + its variance
N_SEEDS=20
FOLD10_STRICT=True    # best_model wins only if it beats M4-alone AND vote-ensemble by > half the CI width
# ---- the 5 source models: (tag, gate-passed selection CSV, feature CSV) ----
SRC_MODELS=[
  ('m1','m1_combined_selection.csv','m1_features.csv'),
  ('m2','m2_combined_selection.csv','m2_features.csv'),
  ('m3','m3_combined_selection.csv','m3_features.csv'),
  ('m4','m4_combined_selection_wavelet_env.csv','m4_features_wavelet_env.csv'),
  ('m5','m5v2_combined_selection.csv','m5v2_features.csv'),
]
meta=pd.read_csv(os.path.join(PROCESSED,'metadata_combined.csv'),dtype={'ecg_id':str})
assert int((meta.label==1).sum())==142, "Expected 142 WPW!"
GATELISTS={}
for tag,self,_ in SRC_MODELS:
    s=pd.read_csv(os.path.join(PROCESSED,self))
    g=s[s['gate']].feature.tolist() if 'gate' in s.columns else s.feature.tolist()
    GATELISTS[tag]=g
print("gate-passed per model:", {t:len(v) for t,v in GATELISTS.items()}, "| union=",sum(len(v) for v in GATELISTS.values()))
print(f"config: shadow {SHADOW_N} | stability {STAB_B}x@{STAB_THR} | gap_cap {GAP_CAP} | nested x{NESTED_REPEATS} | fold10 STRICT={FOLD10_STRICT}")
print("Block bm.0 OK.")


### Block bm.1: Pool assembly (gate-passed union, angle-prefixed, cached)

In [ ]:
# Pool assembly: read ONLY the gate-passed columns from each model's feature CSV, prefix by model tag (uniqueness
# + traceability of which angle each feature comes from), join on ecg_id. Cached to disk (one-time, heavy I/O).
POOL_CSV=os.path.join(PROCESSED,'bestmodel_pool.csv')
if os.path.exists(POOL_CSV):
    print(f"[guard] {os.path.basename(POOL_CSV)} exists -> pool assembly skipped.")
    df=pd.read_csv(POOL_CSV,dtype={'ecg_id':str})
else:
    base=meta[['ecg_id','patient_id','label','fold','source']].copy()
    for tag,_,featf in SRC_MODELS:
        cols=GATELISTS[tag]; t0=time.time()
        usecols=['ecg_id']+cols
        d=pd.read_csv(os.path.join(PROCESSED,featf),usecols=lambda c:c in set(usecols),dtype={'ecg_id':str})
        d=d.rename(columns={c:f'{tag}__{c}' for c in cols})   # prefix
        base=base.merge(d,on='ecg_id',how='left')
        print(f"  {tag}: +{len(cols)} feats from {featf} in {(time.time()-t0)/60:.1f} min | pool now {base.shape[1]-5} feats")
    # downcast features to float32
    fc=[c for c in base.columns if '__' in c]; base[fc]=base[fc].astype('float32')
    base.to_csv(POOL_CSV,index=False); df=base; print(f"  pool written: {df.shape[0]} ECGs x {len(fc)} features")
FEATS_ALL=[c for c in df.columns if '__' in c]
tr=df[df.fold.between(1,8)]; print(f"pool: {len(FEATS_ALL)} features | train1-8 {len(tr)} ({int((tr.label==1).sum())} WPW)")
print("by model:", {tag:sum(c.startswith(tag+'__') for c in FEATS_ALL) for tag,_,_ in SRC_MODELS})
assert int((tr.label==1).sum())>=110, "pool join incomplete?"


### Block bm.2: Cross-model de-duplication (Spearman>0.90) + shadow features

In [ ]:
# Cross-model dedup (Spearman>0.90, ranked by |d| so the strongest of each redundant cluster survives) +
# inject SHADOW_N random null features (if any survives selection -> the pipeline is leaking/overfitting).
def _absd_col(a,b):
    a=a[~np.isnan(a)]; b=b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return 0.0
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return abs((a.mean()-b.mean())/sp) if sp>0 else 0.0
def dedup_fast(feats,thr=0.90):
    if not feats: return []
    sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0); ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
    C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
    for f in feats:
        if all(C[idx[f],idx[k]]<=thr for k in keep): keep.append(f)
    return keep
DEDUP_CSV=os.path.join(PROCESSED,'bestmodel_dedup.csv')
if os.path.exists(DEDUP_CSV):
    dedup_list=pd.read_csv(DEDUP_CSV).feature.tolist(); print(f"[guard] cross-model dedup reloaded ({len(dedup_list)})")
else:
    w=tr[tr.label==1]; n=tr[tr.label==0]; dvals={}
    for c in tqdm(FEATS_ALL,desc='|d| rank',unit='f'): dvals[c]=_absd_col(w[c].values.astype(float),n[c].values.astype(float))
    ordered=sorted(FEATS_ALL,key=lambda c:-dvals[c])
    dedup_list=dedup_fast(ordered,0.90); pd.DataFrame({'feature':dedup_list}).to_csv(DEDUP_CSV,index=False)
from collections import Counter
print(f"cross-model dedup>0.90: {len(FEATS_ALL)} -> {len(dedup_list)} | by model {dict(Counter(c.split('__')[0] for c in dedup_list))}")
rng=np.random.default_rng(0)
SHADOW=[f'shadow__{i}' for i in range(SHADOW_N)]
for s in SHADOW:
    if s not in df.columns: df[s]=rng.standard_normal(len(df)).astype('float32')
tr=df[df.fold.between(1,8)]
CAND=dedup_list+SHADOW
print(f"candidate pool = {len(dedup_list)} real (deduped) + {SHADOW_N} shadow = {len(CAND)}")


### Block bm.3: Stability selection (300×) + ranking par importance-permutation par-fold

In [ ]:
# Modeling matrices + stability selection (300x bootstrap of |d|) + per-fold PERMUTATION-IMPORTANCE ranking
# (catches features useful in INTERACTION, not just univariate |d| -> best_model's whole point).
d8=df[df.fold.between(1,8)].reset_index(drop=True); y=d8.label.values; folds=d8.fold.values; src8=d8.source.values; eid8=d8.ecg_id.values
FOLD_MASKS=[(folds!=h,folds==h) for h in sorted(np.unique(folds))]
f9=df[df.fold==9].reset_index(drop=True); y9=f9.label.values; spw8=(y==0).sum()/max((y==1).sum(),1)
Xall=d8[CAND].to_numpy(np.float32); X9all=f9[CAND].to_numpy(np.float32); COL={c:i for i,c in enumerate(CAND)}
def _ci(feats): return [COL[c] for c in feats]
def make_xgb(spw,**kw):
    p=dict(n_estimators=200,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=2.0,
           min_child_weight=3,scale_pos_weight=spw,eval_metric='aucpr',tree_method='hist',random_state=42,n_jobs=6)
    p.update(kw); return XGBClassifier(**p)
def oof_xgb(feats,**kw):
    X=Xall[:,_ci(feats)]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        oof[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**kw).fit(X[trm],y[trm]).predict_proba(X[vam])[:,1]
    return oof
def oof_ap(feats,**kw):
    oof=oof_xgb(feats,**kw); m=~np.isnan(oof); return float(average_precision_score(y[m],oof[m]))
# stability selection (vectorized bootstrap |d|)
STAB=os.path.join(PROCESSED,'bestmodel_stability.csv')
if os.path.exists(STAB):
    sdf=pd.read_csv(STAB); stab=dict(zip(sdf.feature,sdf.freq)); print("[guard] stability reloaded")
else:
    yk=y.astype(int)
    def col_absd(Xs,ys):
        wv,nv=Xs[ys==1],Xs[ys==0]; mw,mn=np.nanmean(wv,0),np.nanmean(nv,0); vw,vn=np.nanvar(wv,0,ddof=1),np.nanvar(nv,0,ddof=1)
        sp=np.sqrt(((len(wv)-1)*vw+(len(nv)-1)*vn)/max(len(wv)+len(nv)-2,1))+EPS; return np.abs((mw-mn)/sp)
    rngb=np.random.default_rng(7); freq=np.zeros(len(CAND))
    for _ in tqdm(range(STAB_B),desc='stability',unit='boot'):
        idx=rngb.choice(len(yk),len(yk),True); freq+=(np.nan_to_num(col_absd(Xall[idx],yk[idx]))>0.3)
    freq/=STAB_B; stab=dict(zip(CAND,freq)); pd.DataFrame({'feature':CAND,'freq':freq}).to_csv(STAB,index=False)
STABLE=[f for f in dedup_list if stab.get(f,0)>=STAB_THR]
nsh_stab=sum(stab.get(s,0)>=STAB_THR for s in SHADOW)
print(f"stable (>= {STAB_THR}) real features: {len(STABLE)} | SHADOWS surviving stability: {nsh_stab}/{SHADOW_N} (should be ~0)")
# per-fold permutation importance ranking (leak-safe: fit on train, importance on held fold)
IMP=os.path.join(PROCESSED,'bestmodel_importance.csv')
RANKSET=STABLE+SHADOW   # keep shadows in the ranking to see if they sneak up
if os.path.exists(IMP):
    idf=pd.read_csv(IMP); RANKED=idf.feature.tolist(); imp_mean=dict(zip(idf.feature,idf.imp)); print("[guard] importance reloaded")
else:
    ci=_ci(RANKSET); Xs=Xall[:,ci]; imp=np.zeros(len(RANKSET))
    for trm,vam in tqdm(FOLD_MASKS,desc='perm-importance',unit='fold'):
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        mdl=make_xgb(spw8).fit(Xs[trm],y[trm])
        r=permutation_importance(mdl,Xs[vam],y[vam],n_repeats=5,scoring='average_precision',random_state=0,n_jobs=6)
        imp+=r.importances_mean
    order=np.argsort(-imp); RANKED=[RANKSET[i] for i in order]; imp_mean=dict(zip(RANKSET,imp))
    pd.DataFrame({'feature':RANKED,'imp':[imp_mean[f] for f in RANKED]}).to_csv(IMP,index=False)
RANKED_REAL=[f for f in RANKED if not f.startswith('shadow__')]
top_sh=[i for i,f in enumerate(RANKED) if f.startswith('shadow__')]
print(f"importance ranking done. top-15: {RANKED[:15]}")
print(f"  best shadow rank: {min(top_sh) if top_sh else 'none'} (high = good; a shadow in the top = leak alarm)")


### Block bm.4: AP-vs-k (importance-ranked) + K x config grid + selection (gap cap)

In [ ]:
# AP-vs-k on the IMPORTANCE ranking (real features) + K x config grid under gap cap + selection.
RK=os.path.join(PROCESSED,'bestmodel_ap_vs_k.csv')
if os.path.exists(RK): rk=pd.read_csv(RK)
else:
    KMAX_SWEEP=min(600,len(RANKED_REAL)); rows=[]; ks=list(range(2,KMAX_SWEEP+1,10))
    for ii,k in enumerate(tqdm(ks,desc='AP-vs-k',unit='k')):
        rows.append({'k':int(k),'AP_oof':oof_ap(RANKED_REAL[:int(k)])})
        if ii%6==0: pd.DataFrame(rows).to_csv(RK,index=False)
    rk=pd.DataFrame(rows); rk.to_csv(RK,index=False)
plt.figure(figsize=(11,5)); plt.plot(rk.k,rk.AP_oof,'-',color='#2563eb',lw=1,label='best_model AP OOF (importance-ranked)')
for v,c,l in [(0.619,'#ea580c','M3 0.619'),(0.718,'#dc2626','M4 0.718'),(0.736,'#000','M3+M4 vote 0.736')]:
    plt.axhline(v,ls=':',color=c,lw=.9,label=l)
plt.xlabel('k'); plt.ylabel('AP OOF'); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title('best_model — AP vs k [OOF, importance-ranked]')
plt.savefig(os.path.join(FIG,'bestmodel_ap_vs_k.png'),dpi=150,bbox_inches='tight'); plt.show()
kbest=rk.loc[rk.AP_oof.idxmax()]; K_auto=int(rk[rk.AP_oof>=0.95*kbest.AP_oof].k.min())
print(f"max OOF AP {kbest.AP_oof:.3f}@{int(kbest.k)} | K_auto {K_auto} | k_max {len(RANKED_REAL)}")
def diag_one(cfg,feats):
    ci=_ci(feats); X8=Xall[:,ci]; X9=X9all[:,ci]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        oof[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**cfg).fit(X8[trm],y[trm]).predict_proba(X8[vam])[:,1]
    m=~np.isnan(oof); ao=average_precision_score(y[m],oof[m]); mdl=make_xgb(spw8,**cfg).fit(X8,y)
    at=average_precision_score(y,mdl.predict_proba(X8)[:,1]); af=average_precision_score(y9,mdl.predict_proba(X9)[:,1]) if y9.sum()>0 else np.nan
    return ao,at,at-ao,af
G2=os.path.join(PROCESSED,'bestmodel_grid2d.csv'); kmax=min(600,len(RANKED_REAL))
KS=sorted(set(list(range(40,kmax+1,60))+[K_auto])); KS=[k for k in KS if 2<=k<=kmax]
CFGS={}
for dep,lrs in [(2,(0.03,0.05)),(3,(0.03,0.05,0.1)),(4,(0.03,0.05))]:
    for lr in lrs: CFGS[f'd{dep}_lr{int(lr*100):02d}']=dict(max_depth=dep,learning_rate=lr,n_estimators=300,min_child_weight=3)
print(f"KS grid ({len(KS)} pts): {KS}")
if os.path.exists(G2): G2d=pd.read_csv(G2)
else:
    rows=[]
    for k in tqdm(KS,desc='grid',unit='K'):
        for cn,c in CFGS.items():
            ao,at,gp,af=diag_one(c,RANKED_REAL[:k]); rows.append({'K':k,'cfg':cn,'depth':c['max_depth'],'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
    G2d=pd.DataFrame(rows); G2d.to_csv(G2,index=False)
print(G2d.sort_values('AP_oof',ascending=False).head(8)[['K','cfg','depth','AP_oof','AP_train','gap','AP_f9']].to_string(index=False))
raw_best=G2d.loc[G2d.AP_oof.idxmax()]
capped=G2d[G2d.gap<=GAP_CAP]; pool=capped if len(capped)>0 else G2d
mo=float(pool.AP_oof.max()); best=pool[pool.AP_oof>=mo-TIE_EPS].sort_values(['K','depth']).iloc[0]
K=int(best.K); CFGNAME=best['cfg']; FEATURES_K=RANKED_REAL[:K]; CFG={**CFGS[CFGNAME],'subsample':0.8,'colsample_bytree':0.8,'reg_lambda':2.0}
print(f"  raw max-OOF: K={int(raw_best.K)} {raw_best.cfg} OOF {raw_best.AP_oof:.3f} gap {raw_best.gap:+.3f}")
print(f">>> SELECTED (gap<= {GAP_CAP} + parsimony): K={K} {CFGNAME} depth{int(best.depth)} | OOF {best.AP_oof:.3f} fold9 {best.AP_f9:.3f} gap {best.gap:+.3f}")
print(f"  by model in FEATURES_K: {dict(Counter(c.split('__')[0] for c in FEATURES_K))}")


### Block bm.5: Repeated nested CV (unbiased AP + variance = selection optimism)

In [ ]:
# Repeated nested-CV (unbiased AP + variance): inside each outer fold, RE-RANK the stable features by |d| on the
# INNER-train only (no peek at the outer fold), take top-K, fit CFG, predict outer fold. Repeats vary the model
# seed. The gap (selection in-sample) - (nested) = the selection optimism. Pragmatic nested (re-ranks; doesn't
# re-dedup/re-permutation — too costly x folds x reps), documented.
SIDX=[COL[f] for f in STABLE]
def nested_oof(seed):
    oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        Xt=Xall[trm]; yt=y[trm]; wi=Xt[yt==1]; ni=Xt[yt==0]
        mw=np.nanmean(wi,0);mn=np.nanmean(ni,0);vw=np.nanvar(wi,0,ddof=1);vn=np.nanvar(ni,0,ddof=1)
        sp=np.sqrt(((len(wi)-1)*vw+(len(ni)-1)*vn)/max(len(wi)+len(ni)-2,1))+EPS; dabs=np.abs((mw-mn)/sp)
        order=sorted(SIDX,key=lambda j:-dabs[j])[:K]
        m=make_xgb((yt==0).sum()/max((yt==1).sum(),1),**{**CFG,'random_state':seed}).fit(Xt[:,order],yt)
        oof[vam]=m.predict_proba(Xall[vam][:,order])[:,1]
    mk=~np.isnan(oof); return float(average_precision_score(y[mk],oof[mk]))
NST=os.path.join(PROCESSED,'bestmodel_nested.csv')
if os.path.exists(NST): nst=pd.read_csv(NST).ap.tolist(); print("[guard] nested reloaded")
else:
    nst=[nested_oof(s) for s in tqdm(range(NESTED_REPEATS),desc='nested-CV',unit='rep')]
    pd.DataFrame({'rep':range(NESTED_REPEATS),'ap':nst}).to_csv(NST,index=False)
ap_nested=float(np.mean(nst)); ap_nested_sd=float(np.std(nst))
print(f">>> NESTED-CV unbiased OOF AP = {ap_nested:.3f} +/- {ap_nested_sd:.3f}  (in-sample-selected {best.AP_oof:.3f}; optimism {best.AP_oof-ap_nested:+.3f})")


### Block bm.6: Leave-one-dataset-out (external validity)

In [ ]:
# Leave-one-dataset-out external validity of best_model: train one site (folds 1-8), test the OTHER (folds 1-8).
# The real generalization test (harder than internal CV). fold10 + cross fold9/10 still untouched.
ciK=_ci(FEATURES_K)
def lodo(train_src,test_src):
    trm=(src8==train_src); tem=(src8==test_src)
    Xtr=Xall[trm][:,ciK]; ytr=y[trm]; Xte=Xall[tem][:,ciK]; yte=y[tem]
    if ytr.sum()==0 or yte.sum()==0: return np.nan,np.nan
    m=make_xgb((ytr==0).sum()/max((ytr==1).sum(),1),**CFG).fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]
    return float(average_precision_score(yte,p)), float(roc_auc_score(yte,p))
print("=== leave-one-dataset-out (external validity) ===")
for a,b_ in (('ptbxl','ningbo'),('ningbo','ptbxl')):
    ap,auc=lodo(a,b_); print(f"  train {a:6s} -> test {b_:6s}: AP {ap:.3f} | AUC {auc:.3f}  (AP collapse expected = batch effect; AUC should hold)")


### Block bm.7: Fit final + SHAP (angles) + confondants + baseline vote-ensemble + gel

In [ ]:
# Final fit + multiseed + evaluate_standard (fold9) + SHAP (which features/angles drive it) + confounds + freeze.
# Also build the VOTE-ENSEMBLE baseline (logistic on per-model OOF scores) for the fold10 showdown.
X8=Xall[:,ciK]; X9=X9all[:,ciK]
def make_final(**kw): return make_xgb(spw8,**{**CFG,**kw})
xgb_raw=make_final().fit(X8,y); ap_train=average_precision_score(y,xgb_raw.predict_proba(X8)[:,1])
oof_raw=np.full(len(d8),np.nan)
for trm,vam in FOLD_MASKS:
    if y[trm].sum()==0 or y[vam].sum()==0: continue
    oof_raw[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**CFG).fit(X8[trm],y[trm]).predict_proba(X8[vam])[:,1]
mask=~np.isnan(oof_raw); ap_oof=float(average_precision_score(y[mask],oof_raw[mask]))
platt=LogisticRegression(max_iter=2000).fit(oof_raw[mask].reshape(-1,1),y[mask])
aps=[];aucs=[]
for s in range(N_SEEDS):
    m=make_final(random_state=s).fit(X8,y); pp=m.predict_proba(X9)[:,1]; aps.append(average_precision_score(y9,pp)); aucs.append(roc_auc_score(y9,pp))
MULTI=dict(AP_mean=float(np.mean(aps)),AP_std=float(np.std(aps)),AUC_mean=float(np.mean(aucs)),AUC_std=float(np.std(aucs)))
score9=xgb_raw.predict_proba(X9)[:,1]; score9_cal=platt.predict_proba(score9.reshape(-1,1))[:,1]
BM_STD=evaluate_standard(name='best_model',y_oof=y[mask],score_oof=oof_raw[mask],y_test=y9,score_test=score9,
    figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=score9_cal,ap_train=ap_train,multiseed=MULTI,
    extra={'K':K,'cfg':CFGNAME,'OOF_AP':ap_oof,'nested_AP':ap_nested,'nested_AP_sd':ap_nested_sd,'by_model':dict(Counter(c.split('__')[0] for c in FEATURES_K))})
print(f"OOF {ap_oof:.3f} | nested {ap_nested:.3f}+/-{ap_nested_sd:.3f} | fold9 {BM_STD['AP']:.3f} | AUC {BM_STD['AUC']:.3f} | multiseed {MULTI['AP_mean']:.3f}+/-{MULTI['AP_std']:.3f}")
print(f"  dataset-confound AUC(source~score) = {roc_auc_score((src8=='ptbxl').astype(int)[mask],oof_raw[mask]):.3f} (0.5=clean)")
# SHAP — which features / which ANGLES drive best_model
try:
    import shap; sv=shap.TreeExplainer(xgb_raw).shap_values(X8); ma=np.abs(sv).mean(0)
    top=np.argsort(-ma)[:20]; print("  top-20 SHAP:", [(FEATURES_K[i],round(float(ma[i]),4)) for i in top])
    bym={}
    for i,f in enumerate(FEATURES_K): k=f.split('__')[0]; bym[k]=bym.get(k,0)+float(ma[i])
    tot=sum(bym.values())+EPS; print("  SHAP mass by ANGLE:", {k:f'{100*v/tot:.0f}%' for k,v in sorted(bym.items(),key=lambda x:-x[1])})
except Exception as e: print("  SHAP skipped:",type(e).__name__,e)
# vote-ensemble baseline (logistic on per-model OOF scores) — for the fold10 showdown
mm=d8[['ecg_id','label']].copy(); have=[]
for tag,p in [('m1','m1_combined_oof.csv'),('m3','m3_combined_oof.csv'),('m4','m4_combined_oof.csv'),('m5','m5v2_combined_oof.csv')]:
    pp=os.path.join(PROCESSED,p)
    if os.path.exists(pp): mm=mm.merge(pd.read_csv(pp,dtype={'ecg_id':str})[['ecg_id','proba_raw']].rename(columns={'proba_raw':tag}),on='ecg_id',how='left'); have.append(tag)
sub=mm.dropna(subset=have); vote=LogisticRegression(max_iter=1000).fit(sub[have],sub.label)
ap_vote_oof=average_precision_score(sub.label,vote.predict_proba(sub[have])[:,1])
print(f"  vote-ensemble OOF ({'+'.join(have)}) AP {ap_vote_oof:.3f} | best_model OOF {ap_oof:.3f}")
oof_df=d8[['ecg_id','patient_id','fold','source','label']].copy(); oof_df['proba_raw']=oof_raw
oof_df.to_csv(os.path.join(PROCESSED,'bestmodel_oof.csv'),index=False)
joblib.dump(xgb_raw,os.path.join(MODELS,'bestmodel_model_raw.joblib')); joblib.dump(platt,os.path.join(MODELS,'bestmodel_platt.joblib'))
pd.Series(FEATURES_K,name='feature').to_csv(os.path.join(MODELS,'bestmodel_features.csv'),index=False)
json.dump({'model':'best_model','K':K,'cfg':CFGNAME,'OOF_AP':ap_oof,'nested_AP':ap_nested,'nested_AP_sd':ap_nested_sd,
    'fold9':BM_STD,'multiseed':MULTI,'vote_oof_AP':float(ap_vote_oof),'by_angle':dict(Counter(c.split('__')[0] for c in FEATURES_K)),
    'date_frozen':datetime.now().strftime('%Y-%m-%d %H:%M')},open(os.path.join(MODELS,'bestmodel_config.json'),'w'),indent=2)
print("best_model FROZEN -> models/best_model. fold10 still UNTOUCHED.")


### Block bm.8: FOLD 10 — RESERVED (untouched here; final evaluation deferred to the final ensemble evaluation)

In [ ]:
# ===== FOLD 10 — RESERVED, NOT TOUCHED HERE =====
# We do NOT evaluate it in this notebook. fold10 stays a pristine reserve for the SINGLE final evaluation of the
# chosen deployable model (the M3+M4 ensemble), done later in the official ensemble notebook, ideally
# after M7. best_model's verdict rests on the NESTED CV (0.613 < vote 0.685) — fold10 was only a confirmation.
# The full showdown code is kept (b8.py) for reuse in the final ensemble step.
print("fold10 RESERVED — not touched. Final evaluation deferred to the official ensemble notebook.")